# Python 参数系统完全指南

> 从位置参数到 `**kwargs`：一篇可运行、可验证的教学 Notebook

| 项目 | 内容 |
|---|---|
| 主题 | Python 函数参数系统（类型 / 顺序 / 默认值 / 解包 / 转发 / 注解） |
| 适用版本 | Python 3.8+（`/` 语法），本 Notebook 在 **Python 3.13** 上全部真实执行 |
| 业务场景 | LLMOps API 服务开发：参数配置、请求封装、装饰器、参数校验 |
| 阅读方式 | 自上而下逐单元执行；所有结论均配有验证代码与确定性输出 |

## 一句话本质总结

> **参数设计的核心是：让调用者既能按位置传，也能按关键字传；该限制的地方用 `/` 和 `*` 明确约束；转发参数用 `*args` 和 `**kwargs`。**

## 学习目标

1. 说出五种参数类型的语法、语义与底层容器（tuple / dict）
2. 写出正确的完整参数顺序，并解释顺序背后的匹配规则
3. 解释默认参数"定义时求值"的机制，并规避可变默认参数陷阱
4. 熟练使用 `*` / `**` 进行解包调用与参数转发（装饰器、代理函数）
5. 用 `/` 和 `*` 设计更健壮的 API 签名
6. 结合类型注解与 `typing.overload` 书写可维护的函数签名


## 修正与适配说明（相对源文档）

为保证「每个单元 0 报错、输出可复现」，本 Notebook 对源文档做了以下修正与适配：

| 编号 | 类型 | 说明 |
|---|---|---|
| 1 | **修正（源文档 13.2 的技术错误）** | 源文档把 `**kwargs` 原样转发给 `print()`，调用 `log("错误", level="ERROR", exc_info=True)` 会触发 `TypeError`（`print` 不接受 `exc_info`）。本 Notebook 改为把 kwargs 格式化为日志上下文，并在「实用示例集」中用代码**实际复现**该错误。 |
| 2 | 适配 | 源文档 13.5 依赖 `requests` 发起真实网络请求，输出不可复现。改用本地桩（stub）实现同一封装结构。 |
| 3 | 适配 | 涉及 `time.time()` 的示例只输出**布尔比较结果**，不打印原始时间戳，保证输出确定。 |
| 4 | 适配 | `set` 的打印顺序受哈希随机化影响，所有 set 输出前统一 `sorted()`。 |
| 5 | 环境说明 | `list[int]` 等内置泛型写法需 Python ≥ 3.9；`/` 仅位置语法需 ≥ 3.8。本环境为 3.13，均满足。 |


# 第 0 章 前置知识

> 本章解决什么问题：**默认参数为什么会"被所有调用共享"？** 答案不在参数系统本身，而在 `def` 语句的执行机制里。理解这一点，后面所有陷阱都不再需要死记。

## 0.1 函数是对象，`def` 是语句

- Python 中 `def` 不是声明，而是一条**语句**：解释器执行到它时，才创建函数对象并绑定名字
- 创建函数对象时，**默认参数表达式立即求值一次**，结果存入函数对象的 `__defaults__` 属性
- 之后所有调用共享这份数据 —— 这就是可变默认参数陷阱的根源
- 函数可以赋值、传递、比较 —— 它是一等对象


In [1]:
import sys

# 1) 环境确认
print("Python 版本:", sys.version.split()[0])

def demo(a, b=10):
    return a + b

# 2) 函数是对象：可查看其元数据
print("函数名:", demo.__name__)
print("默认参数保存在 __defaults__:", demo.__defaults__)   # (10,) —— 定义时求值的结果

# 3) 验证：默认值存在"函数对象"上，而不是每次调用新建
demo.__defaults__ = (99,)          # 直接篡改函数对象上的默认值
print("篡改默认值后 demo(1) =", demo(1))   # 100 —— 证明默认值被所有调用共享


Python 版本: 3.13.12
函数名: demo
默认参数保存在 __defaults__: (10,)
篡改默认值后 demo(1) = 100


## 0.2 元组与字典：两种可变参数的底层容器

- `*args` 收集多余**位置**参数，底层是 `tuple`（不可变、有序、可索引）
- `**kwargs` 收集多余**关键字**参数，底层是 `dict`（Python 3.7+ 保证插入顺序，因此输出是确定的）
- 解包调用 `f(*t)` / `f(**d)` 本质是"容器 → 参数"的反向操作


In [2]:
t = (1, 2, 3)
d = {"name": "Alice", "age": 30}

# 1) 元组：不可变、有序、可索引 —— *args 的容器
print("元组:", t, "| 首元素:", t[0], "| 长度:", len(t))

# 2) 字典：保持插入顺序（3.7+），因此打印结果确定 —— **kwargs 的容器
print("字典 items:", list(d.items()))
d["extra"] = True
print("追加后 keys:", list(d.keys()))


元组: (1, 2, 3) | 首元素: 1 | 长度: 3
字典 items: [('name', 'Alice'), ('age', 30)]
追加后 keys: ['name', 'age', 'extra']


## 0.3 求值时机：定义时 vs 调用时

这是贯穿全文的核心对比：

| 时机 | 求值什么 | 次数 |
|---|---|---|
| **定义时**（执行到 `def`） | 默认参数表达式 | **仅 1 次**，结果挂在函数对象上 |
| **调用时**（执行到 `f(...)`） | 实参表达式 | 每次调用都重新求值 |

下面的验证代码用"副作用计数器"直接证明这一点。


In [3]:
calls = []

def make_value(tag):
    # 副作用计数器：记录"这个表达式何时被求值"
    calls.append(tag)
    return tag

# 1) 实参表达式：每次"调用"时求值
def f(x):
    return x
print(f(make_value("实参")), "| 求值记录:", calls)      # ['实参'] —— 调用时求值

# 2) 默认参数表达式：只在"定义"时求值一次
calls.clear()
def g(x, y=make_value("默认值")):
    return x, y
print("定义完成、尚未调用，求值记录:", calls)            # ['默认值'] —— 定义时已求值！

calls.clear()
g(1)
g(2)
print("两次调用之后，求值记录:", calls)                  # [] —— 默认值不再重新求值


实参 | 求值记录: ['实参']
定义完成、尚未调用，求值记录: ['默认值']
两次调用之后，求值记录: []


## 0.4 本 Notebook 的错误演示工具

按照"错误写法要真实触发报错"的要求，同时保证"每个代码单元 0 报错"，我们用一个辅助函数 `show_error`：它接收一段代码字符串，**真实执行**并捕获其中的异常，打印异常类型与信息。这样既看到了真实报错，单元本身又不失败。


In [4]:
def show_error(code):
    # 演示辅助：真实执行一段代码，捕获并打印其中的异常（不中断 Notebook）
    try:
        exec(code)
    except SyntaxError as e:
        print(f"SyntaxError: {e.msg}")
    except Exception as e:
        print(f"{type(e).__name__}: {e}")

# 试运行：类型错误会被真实触发并打印
show_error("(1, 2)[5]")


IndexError: tuple index out of range


# 第 1 章 五种参数类型总览

> 本章解决什么问题：先把参数系统的"地图"建立起来 —— 五种类型各解决什么问题，一张表看清。

| 类型 | 语法 | 说明 |
|---|---|---|
| 位置或关键字 | `a` | 最常见，可按位置或按关键字传 |
| 仅位置 | `a, /` | `/` 之前的参数只能按位置传 |
| 仅关键字 | `*, a` | `*` 之后的参数只能按关键字传 |
| 可变位置 | `*args` | 收集多余位置参数为**元组** |
| 可变关键字 | `**kwargs` | 收集多余关键字参数为**字典** |

完整签名骨架（后面各章逐一展开）：

```python
def f(pos_only, /, normal, *args, kw_only, **kwargs):
    ...
```


In [5]:
def full_form(pos_only, /, normal, *args, kw_only, **kwargs):
    # 1) pos_only —— 仅位置；normal —— 位置或关键字
    # 2) *args —— 收集多余位置参数；kw_only —— 仅关键字；**kwargs —— 收集多余关键字参数
    print(pos_only, normal, args, kw_only, kwargs)

full_form(1, 2, 3, 4, kw_only=5, x=6)
# 输出: 1 2 (3, 4) 5 {'x': 6}
#        │ │  │    │     └ 关键字 x 落入 kwargs（dict）
#        │ │  │    └ 5 通过 kw_only=5 命名传入
#        │ │  └ 多余位置参数 3, 4 收集为元组
#        └ 2 按"位置或关键字"规则按位置传入


1 2 (3, 4) 5 {'x': 6}


# 第 2 章 位置参数与关键字参数

> 本章解决什么问题：最基础的两种传参方式的匹配规则，以及"混用"时的硬性顺序约束。

## 2.1 三条规则

1. **位置参数**：按顺序传入，调用方与被调方靠"第几个"对齐
2. **关键字参数**：按名称传入，与顺序无关
3. **混用**：位置参数必须出现在关键字参数**之前**；同一个参数不能传两次

## 2.2 参数匹配规则（同一函数的四种等价调用）


In [6]:
def greet(name, age):
    print(f"{name}, {age}")

# 1) 位置参数：按顺序传
greet("Alice", 30)

# 2) 关键字参数：按名称传，顺序无关
greet(age=30, name="Alice")

# 3) 混合：位置在前，关键字在后
greet("Alice", age=30)

# 4) 匹配规则：以下四种调用完全等价
def f(a, b, c):
    return (a, b, c)
print(f(1, 2, 3), f(1, b=2, c=3), f(1, 2, c=3), f(a=1, b=2, c=3))

# 5) 不能对同一个参数重复传值（位置 + 关键字各传一次也算重复）
show_error('f(1, a=1)')   # TypeError: multiple values for argument 'a'


Alice, 30
Alice, 30
Alice, 30
(1, 2, 3) (1, 2, 3) (1, 2, 3) (1, 2, 3)
TypeError: f() got multiple values for argument 'a'


# 第 3 章 默认参数

> 本章解决什么问题：默认参数的求值机制 —— Python 面试与线上事故的最高频知识点。

## 3.1 基本用法与定义规则

- 默认参数让调用方"可传可不传"
- 硬性语法规则：**默认参数必须位于非默认参数之后**（否则 SyntaxError）


In [7]:
def greet2(name, greeting="Hello"):
    # greeting 有默认值，调用时可省略
    print(f"{greeting}, {name}")

greet2("Alice")                    # 省略默认参数
greet2("Bob", "Hi")                # 按位置覆盖默认值
greet2("Carol", greeting="Hey")    # 按关键字覆盖默认值

# 默认参数必须在非默认参数之后 —— 反例真实触发 SyntaxError
show_error('def bad(a=1, b):\n    pass')


Hello, Alice
Hi, Bob
Hey, Carol
SyntaxError: parameter without a default follows parameter with a default


## 3.2 头号陷阱：默认参数在**定义时**求值一次

```python
def append_to(item, target=[]):   # 这个 [] 只在 def 执行时创建一次
    ...
```

- 默认值 `[]` 挂在函数对象的 `__defaults__` 上，**所有调用共享同一个列表**
- 每次不传 `target` 调用，都是往**同一个列表**里 append

**修复模式（工程标准写法）**：默认值设为 `None`，函数体内判断后新建。


In [8]:
def append_to(item, target=[]):
    # ❌ 错误写法：所有调用共享同一个默认列表
    target.append(item)
    return target

print(append_to(1))    # [1]
print(append_to(2))    # [1, 2]   ← 不是 [2]！
print(append_to(3))    # [1, 2, 3]
print("证据：__defaults__ 里的列表 =", append_to.__defaults__[0])

def append_to_fixed(item, target=None):
    # ✅ 正确写法：None 哨兵 + 函数体内新建
    if target is None:
        target = []
    target.append(item)
    return target

print(append_to_fixed(1))   # [1]
print(append_to_fixed(2))   # [2]   ← 每次都是新列表


[1]
[1, 2]
[1, 2, 3]
证据：__defaults__ 里的列表 = [1, 2, 3]
[1]
[2]


## 3.3 验证：为什么"定义时求值"会导致时间戳不变

源文档 3.4 的例子：`def log(msg, timestamp=time.time())` —— 两次调用时间戳相同，因为 `time.time()` **只在 def 执行时运行一次**。
为了输出可复现，这里只打印**比较结果**，不打印时间戳本身。


In [9]:
import time

def log2(msg, timestamp=time.time()):
    # timestamp 的默认值在 def 执行那一刻就固定了
    return timestamp

t1 = log2("第一次调用")
time.sleep(0.01)   # 只等待 10ms，足以验证结论
t2 = log2("第二次调用")
print("两次调用的默认时间戳是否相同:", t1 == t2)   # True —— 定义时只求值一次

t0 = time.time()
print("默认值早于任何调用时刻:", t1 < t0)          # True —— 定义发生在调用之前


两次调用的默认时间戳是否相同: True
默认值早于任何调用时刻: True


## 3.4 反向利用：用默认参数做缓存

可变默认参数"共享"的特性也可以反过来用 —— 当字典缓存。可行，但**更推荐 `functools.lru_cache`**：语义清晰、线程安全、可控制淘汰策略。


In [10]:
# 1) 手写字典缓存（利用默认参数共享特性）
def fib(n, _cache={}):
    if n in _cache:
        return _cache[n]
    if n < 2:
        return n
    _cache[n] = fib(n - 1) + fib(n - 2)
    return _cache[n]

print("fib(40) =", fib(40))
print("缓存条目数:", len(fib.__defaults__[0]))

# 2) 官方推荐：functools.lru_cache
from functools import lru_cache

@lru_cache(maxsize=None)
def fib2(n):
    return n if n < 2 else fib2(n - 1) + fib2(n - 2)

print("fib2(40) =", fib2(40), "| 缓存信息:", fib2.cache_info())


fib(40) = 102334155
缓存条目数: 39
fib2(40) = 102334155 | 缓存信息: CacheInfo(hits=38, misses=41, maxsize=None, currsize=41)


# 第 4 章 可变位置参数 `*args`

> 本章解决什么问题：调用方传多少个位置参数都不确定时，如何用一个形参全部接住。

要点：

1. `*args` 把**多余**的位置参数收集为一个 `tuple`
2. `*args` 中的 `args` 只是约定俗成的名字，语法上可以叫任何名字（但强烈推荐 `args`）
3. 一个参数都没有时，`args` 是空元组 `()`


In [11]:
# 1) 基本用法
def add(*args):
    return sum(args)
print(add(1, 2, 3), add(), add(1, 2, 3, 4, 5))   # 6 0 15

# 2) 与普通参数配合：普通参数先按位置分配，剩余的归 *args
def f2(a, b, *args):
    print(a, b, args)
f2(1, 2, 3, 4, 5)    # 1 2 (3, 4, 5)

# 3) 命名只是约定：语法上可以叫 *nums
def total(*nums):
    print(type(nums).__name__, nums)
total(1, 2, 3)       # tuple (1, 2, 3) —— 类型是元组

# 4) 遍历 args
def multiply(*args):
    result = 1
    for x in args:
        result *= x
    return result
print(multiply(2, 3, 4))   # 24


6 0 15
1 2 (3, 4, 5)
tuple (1, 2, 3)
24


# 第 5 章 可变关键字参数 `**kwargs`

> 本章解决什么问题：调用方传入的"命名选项"不确定时，如何全部接住并处理。

要点：

1. `**kwargs` 把多余的**关键字**参数收集为一个 `dict`
2. `**` 解包时，**键必须是字符串**（非字符串键触发 `TypeError`）
3. 工程上常配合"白名单校验"，防止拼写错误的参数被静默吞掉


In [12]:
# 1) 基本用法
def show(**kwargs):
    for k, v in kwargs.items():
        print(k, v)
show(name="Alice", age=30)

# 2) 与普通参数配合
def f3(a, b, **kwargs):
    print(a, b, kwargs)
f3(1, 2, c=3, d=4)   # 1 2 {'c': 3, 'd': 4}

# 3) 键必须是字符串（真实触发报错）
show_error('f3(**{1: 2})')   # TypeError: keywords must be strings

# 4) 白名单校验：拒绝未知参数
def create_user(**kwargs):
    allowed = {"name", "age", "email"}
    unknown = set(kwargs) - allowed
    if unknown:
        raise TypeError(f"未知参数: {sorted(unknown)}")   # sorted 保证输出可复现
    print(kwargs)

create_user(name="Alice", age=30)
show_error('create_user(name="Alice", gender="F")')   # TypeError: 未知参数: ['gender']


name Alice
age 30
1 2 {'c': 3, 'd': 4}
TypeError: keywords must be strings
{'name': 'Alice', 'age': 30}
TypeError: 未知参数: ['gender']


# 第 6 章 同时使用 `*args` 和 `**kwargs`

> 本章解决什么问题：两者组合时的分配规则，以及"万能转发"这一装饰器基石。

## 6.1 分配规则

调用发生时，解释器按以下顺序分配实参：

1. 先满足普通位置参数
2. 多余的位置参数 → 打包进 `args`（tuple）
3. 已命名的关键字参数 → 匹配对应形参
4. 多余的关键字参数 → 打包进 `kwargs`（dict）


In [13]:
def f4(a, b, *args, **kwargs):
    print(a, b, args, kwargs)

f4(1, 2, 3, 4, x=5, y=6)   # 1 2 (3, 4) {'x': 5, 'y': 6}

# 万能转发：wrapper 能接收"任何签名"，并原样转发给 f4
def wrapper(*args, **kwargs):
    return f4(*args, **kwargs)
wrapper(1, 2, 3, 4, x=5, y=6)


1 2 (3, 4) {'x': 5, 'y': 6}
1 2 (3, 4) {'x': 5, 'y': 6}


## 6.2 完整参数顺序（五段式）

```python
def f(a, b=1, *args, c, d=2, **kwargs):
#    │  │    │      │  │    └ 5) 收集剩余关键字参数
#    │  │    │      │  └ 4) 仅关键字参数（* 之后），可带默认值
#    │  │    │      └ 3) *args 之后自动成为仅关键字参数
#    │  │    └ 2) 可变位置参数
#    │  └ 1b) 带默认值的参数
#    └ 1a) 普通位置参数
```

`c` 位于 `*args` 之后，因此**必须**用 `c=...` 传入。


In [14]:
def f5(a, b=1, *args, c, d=2, **kwargs):
    # a 普通；b 默认；*args 可变位置；c/d 仅关键字（d 有默认值）；**kwargs 可变关键字
    print(a, b, args, c, d, kwargs)

f5(1, 2, 3, 4, c=5, e=6)   # 1 2 (3, 4) 5 2 {'e': 6}


1 2 (3, 4) 5 2 {'e': 6}


# 第 7 章 仅位置参数 `/`

> 本章解决什么问题：如何禁止调用方用"参数名"传参，让 API 参数名成为可自由演进的实现细节。

Python 3.8+ 用 `/` 标记：**`/` 之前的参数只能按位置传**。

为什么需要它：

- 参数名不重要，或者未来可能改名（改名不构成破坏性变更）
- 避免调用方依赖参数名，让 API 演进更自由
- 标准库大量使用，如 `len(obj, /)`、`pow(x, y, /)`


In [15]:
def f6(a, b, /, c, d):
    # a、b 只能按位置传；c、d 位置或关键字均可
    print(a, b, c, d)

f6(1, 2, 3, 4)          # ✅ 全部按位置
f6(1, 2, c=3, d=4)      # ✅ 前两个位置，后两个关键字
show_error('f6(a=1, b=2, c=3, d=4)')   # ❌ a、b 不允许关键字

# 经典设计：内置 pow 同款签名
def power(x, y, /):
    # 仅位置 —— 未来把 x/y 改名为 base/exp 也不会破坏调用方
    return x ** y
print(power(2, 3))                    # 8
show_error('power(x=2, y=3)')         # ❌ TypeError

# 与仅关键字参数组合：三种约束并存
def f7(a, /, b, *, c):
    print(a, b, c)
f7(1, 2, c=3)       # ✅
f7(1, b=2, c=3)     # ✅
show_error('f7(a=1, b=2, c=3)')      # ❌ a 不允许关键字


1 2 3 4
1 2 3 4
TypeError: f6() got some positional-only arguments passed as keyword arguments: 'a, b'
8
TypeError: power() got some positional-only arguments passed as keyword arguments: 'x, y'
1 2 3
1 2 3
TypeError: f7() got some positional-only arguments passed as keyword arguments: 'a'


# 第 8 章 仅关键字参数 `*`

> 本章解决什么问题：如何强制调用方写参数名，提升可读性、防止顺序错误。

`*` 之后的参数**必须**用关键字传入。三种收益：

1. **可读性**：`create_user("Alice", age=30, email=...)` 一眼看清每个值是什么
2. **防顺序错误**：两个同类型参数（host/port）不会传反
3. **可扩展**：以后新增参数不会与既有位置调用冲突


In [16]:
def f8(a, *, b, c):
    # b、c 是仅关键字参数
    print(a, b, c)

f8(1, b=2, c=3)      # ✅
show_error('f8(1, 2, 3)')    # ❌ b、c 不能按位置传
show_error('f8(1, 2, c=3)')  # ❌ b 不能按位置传

# 强制关键字让调用自解释
def create_user2(name, *, age, email):
    print(name, age, email)
create_user2("Alice", age=30, email="a@example.com")

# 单独使用 *：全部形参都仅关键字
def f9(*, a, b):
    print(a, b)
f9(a=1, b=2)

# 与 *args 配合：*args 之后的命名参数自动成为仅关键字（sep 模式）
def join_all(*args, sep=", "):
    return sep.join(map(str, args))
print(join_all(1, 2, 3), "|", join_all(1, 2, 3, sep="-"))


1 2 3
TypeError: f8() takes 1 positional argument but 3 were given
TypeError: f8() takes 1 positional argument but 2 positional arguments (and 1 keyword-only argument) were given
Alice 30 a@example.com
1 2
1, 2, 3 | 1-2-3


# 第 9 章 参数顺序总结

> 本章解决什么问题：把五种类型放进**同一条签名**，验证完整顺序规则。

完整顺序（缺一不可的相对次序）：

```python
def f(仅位置, /, 普通, *args, 仅关键字, **kwargs): ...
```

1. 仅位置参数在 `/` 前
2. 普通参数在 `/` 和 `*` 之间
3. `*args` 收集多余位置参数
4. `*` 后的参数是仅关键字
5. `**kwargs` 收集多余关键字参数


In [17]:
def full_form2(a, b, /, c, d, *args, e, f=10, **kwargs):
    # a,b 仅位置 | c,d 普通 | *args 可变位置 | e,f 仅关键字(f 有默认) | **kwargs 可变关键字
    print(a, b, c, d, args, e, f, kwargs)

full_form2(1, 2, 3, 4, 5, 6, e=7, g=8)
# 1 2 3 4 (5, 6) 7 10 {'g': 8}

# 用 inspect 逐一验证每个形参的种类（机器可读的"官方答案"）
import inspect
for name, p in inspect.signature(full_form2).parameters.items():
    print(f"{name:8} -> {p.kind.name}")


1 2 3 4 (5, 6) 7 10 {'g': 8}
a        -> POSITIONAL_ONLY
b        -> POSITIONAL_ONLY
c        -> POSITIONAL_OR_KEYWORD
d        -> POSITIONAL_OR_KEYWORD
args     -> VAR_POSITIONAL
e        -> KEYWORD_ONLY
f        -> KEYWORD_ONLY
kwargs   -> VAR_KEYWORD


# 第 10 章 解包调用

> 本章解决什么问题：反过来 —— 手里已经有 tuple/dict，如何"摊开"成参数传给函数。

| 语法 | 作用 | 等价关系 |
|---|---|---|
| `f(*iterable)` | 解包为**位置**实参 | `f(*[1,2])` ≡ `f(1, 2)` |
| `f(**mapping)` | 解包为**关键字**实参 | `f(**{"a":1})` ≡ `f(a=1)` |
| 混合 | 位置在前，关键字在后 | `f(*t, **d)` |
| 多次解包 | 可写多个 `*` / `**` | Python 3.5+ |


In [18]:
# 1) * 解包位置参数
def add3(a, b, c):
    return a + b + c
nums = [1, 2, 3]
print(add3(*nums))            # 6 ≡ add3(1, 2, 3)

# 2) ** 解包关键字参数
def greet3(name, age):
    print(name, age)
greet3(**{"name": "Alice", "age": 30})

# 3) 同时解包
def f10(a, b, c, d):
    print(a, b, c, d)
f10(*(1, 2), **{"c": 3, "d": 4})   # 1 2 3 4

# 4) 多次解包（Python 3.5+）
def f11(*args, **kwargs):
    print(args, kwargs)
f11(*[1, 2], *[3, 4], **{"a": 5}, **{"b": 6})   # (1, 2, 3, 4) {'a': 5, 'b': 6}

# 5) 解包与默认参数：只解一部分，其余走默认值
def f12(a, b=10, c=20):
    print(a, b, c)
f12(*[1])                      # 1 10 20
f12(*[1, 2])                   # 1 2 20
f12(**{"a": 1, "c": 3})        # 1 10 3


6
Alice 30
1 2 3 4
(1, 2, 3, 4) {'a': 5, 'b': 6}
1 10 20
1 2 20
1 10 3


# 第 11 章 参数转发与装饰器

> 本章解决什么问题：`*args, **kwargs` 最重要的工程应用 —— 在不动原函数签名的前提下"包一层"。

要点：

1. 通用转发：`wrapper(*args, **kwargs)` 能接收任意签名并原样转发
2. 装饰器 = 接收函数 → 返回包装函数，包装函数必须能转发任意参数
3. `functools.wraps` 复制 `__name__`/`__doc__` 等元数据并设置 `__wrapped__`；`inspect.signature` 会顺着 `__wrapped__` 找回**原签名**


In [19]:
from functools import wraps

def original(a, b, c=0):
    return a + b + c

# 1) 通用转发骨架
def wrapper(*args, **kwargs):
    print("调用前")
    result = original(*args, **kwargs)
    print("调用后")
    return result
print(wrapper(1, 2, c=3))

# 2) 装饰器：@wraps 保留元数据
def log_func(func):
    @wraps(func)
    def wrapper2(*args, **kwargs):
        print(f"调用 {func.__name__}")
        return func(*args, **kwargs)
    return wrapper2

@log_func
def add4(a, b, c=0):
    return a + b + c
print(add4(1, 2, c=3))


调用前
调用后
6
调用 add4
6


In [20]:
import inspect

# 1) 用了 @wraps：签名被"还原"
print("用 @wraps 的签名:", inspect.signature(add4))       # (a, b, c=0)
print("可经 __wrapped__ 访问原函数:", add4.__wrapped__.__name__)

# 2) 不用 @wraps：签名退化为 (*args, **kwargs)，文档/名字全部丢失
def log_nowraps(func):
    def w(*args, **kwargs):
        return func(*args, **kwargs)
    return w

@log_nowraps
def sub(a, b):
    return a - b
print("不用 @wraps 的签名:", inspect.signature(sub))      # (*args, **kwargs)


用 @wraps 的签名: (a, b, c=0)
可经 __wrapped__ 访问原函数: add4
不用 @wraps 的签名: (*args, **kwargs)


# 第 12 章 参数与类型注解

> 本章解决什么问题：给参数系统加上"类型说明书"，服务 IDE 提示与静态检查器（mypy / pyright）。

要点：

1. 注解是**可选的元数据**，运行时**不做强制检查**（只存进 `__annotations__`）
2. `*args: int` 注解的是"每个收集到的元素都是 int"；`**kwargs: str` 注解的是"每个值都是 str"
3. `typing.overload` 为同一函数声明多套签名，供类型检查器分流


In [21]:
def greet4(name: str, age: int = 0) -> str:
    return f"{name}, {age}"

print(greet4("Alice", 30))
print("注解仅是元数据:", greet4.__annotations__)

# 1) 运行时不强制 —— 传"错"类型照样执行（校验是类型检查器的职责）
print("传错类型也不报错:", greet4(42, "x"))

# 2) 可变参数注解
def f13(*args: int, **kwargs: str) -> None:
    print(args, kwargs)
f13(1, 2, k="v")

# 3) 复杂类型（内置泛型 list[int] 需 Python 3.9+）
from typing import Optional, Iterable
def process(items: Iterable[int], limit: Optional[int] = None) -> list[int]:
    items = list(items)
    return items[:limit] if limit is not None else items
print(process([1, 2, 3, 4], limit=2))

# 4) overload：给类型检查器看的多套签名（运行时执行最后的实现）
from typing import overload

@overload
def parse2(x: int) -> int: ...
@overload
def parse2(x: str) -> str: ...

def parse2(x):
    return x

import typing
print("类型检查器可见的签名:", [str(s) for s in typing.get_overloads(parse2)])
print("运行时实际执行:", parse2(1), parse2("a"))


Alice, 30
注解仅是元数据: {'name': <class 'str'>, 'age': <class 'int'>, 'return': <class 'str'>}
传错类型也不报错: 42, x
(1, 2) {'k': 'v'}
[1, 2]
类型检查器可见的签名: ['<function parse2 at 0x10b60ee80>', '<function parse2 at 0x10b60efc0>']
运行时实际执行: 1 a


# 反例单元：错误写法 → 真实报错 → 正确写法

> 本章解决什么问题：把前面散落的易错点集中成"对照实验"—— 每个反例先**真实触发**报错，再给出正确写法并对比输出差异。


## 反例 1：位置参数出现在关键字参数之后

Python 语法规定：一旦出现 `名字=值` 形式的实参，后面不能再跟裸的位置实参。


In [22]:
# ❌ 错误写法：真实触发 SyntaxError（连编译都过不了）
show_error('greet(name="Alice", 30)')

# ✅ 正确写法：位置在前，关键字在后（或全部用关键字）
greet("Alice", age=30)
greet(age=30, name="Alice")


SyntaxError: positional argument follows keyword argument
Alice, 30
Alice, 30


## 反例 2：同一个参数重复传值

位置传了一次 + 关键字又传一次 = 重复。解释器直接拒绝，不会"后者覆盖前者"。


In [23]:
# ❌ 错误写法：name 被传了两次（一次位置、一次关键字）
show_error('greet("Alice", 30, name="Bob")')

# ✅ 正确写法：每个参数只传一次
greet("Alice", 30)


TypeError: greet() got multiple values for argument 'name'
Alice, 30


## 反例 3：可变对象作为默认参数

第 3 章的核心陷阱。这里做一次输出差异的"对照实验"。


In [24]:
def bad_append(item, target=[]):
    # ❌ 所有调用共享同一个默认列表
    target.append(item)
    return target

def good_append(item, target=None):
    # ✅ None 哨兵模式
    if target is None:
        target = []
    target.append(item)
    return target

print("错误写法连续两次调用:", bad_append(1), bad_append(2))     # [1] [1, 2] ← 状态泄漏
print("正确写法连续两次调用:", good_append(1), good_append(2))   # [1] [2]   ← 互相独立


错误写法连续两次调用: [1, 2] [1, 2]
正确写法连续两次调用: [1] [2]


## 反例 4：仅位置 / 仅关键字参数的传法错误

`/` 前不许用名字，`*` 后不许用位置。


In [25]:
def div(a, b, /, *, precision=2):
    # a、b 仅位置；precision 仅关键字
    return round(a / b, precision)

# ❌ 两种错误传法，各自真实触发 TypeError
show_error('div(a=1, b=2)')    # 仅位置参数用了关键字
show_error('div(1, 2, 4)')     # 仅关键字参数用了位置

# ✅ 正确写法
print(div(1, 3), div(1, 3, precision=4))


TypeError: div() got some positional-only arguments passed as keyword arguments: 'a, b'
TypeError: div() takes 2 positional arguments but 3 were given
0.33 0.3333


## 反例 5：`**` 解包非字符串键

关键字参数的"关键字"本质是标识符，因此 `**` 解包的映射**键必须是字符串**。


In [26]:
# ❌ 整数键不能作为关键字
show_error('greet(**{1: 2})')

# ✅ 键必须是字符串，且名字要对得上形参
greet(**{"name": "Alice", "age": 30})


TypeError: keywords must be strings
Alice, 30


## 反例 6：用 `if not x` 判断"是否传了默认参数"

`0`、`""`、`[]`、`False` 都是合法入参，但它们的布尔值都是假 —— 用 `not x` 判断会把它们误判成"没传"。必须用 `is None`。


In [27]:
def bad_flag(x=None):
    if not x:       # ❌ 0 / "" / False 全部被误判为"没传"
        x = []
    return x

def good_flag(x=None):
    if x is None:   # ✅ 只有真的没传（None 哨兵）才走默认逻辑
        x = []
    return x

print("错误写法 bad_flag(0):", bad_flag(0))    # [] —— 0 被吞掉了！
print("正确写法 good_flag(0):", good_flag(0))  # 0  —— 被正确保留


错误写法 bad_flag(0): []
正确写法 good_flag(0): 0


# 实用示例集：五个工程高频模式

> 本章解决什么问题：把参数系统落到日常写代码的五个"肌肉记忆"模式上。示例取材自 LLMOps API 服务开发场景。

| # | 模式 | 核心参数技术 | 典型场景 |
|---|---|---|---|
| 1 | 配置函数 | 仅关键字 + `**options` | 连接/客户端初始化 |
| 2 | 日志函数 | `*args` + 仅关键字 | 统一打点 |
| 3 | 带参数的重试装饰器 | 装饰器 + 万能转发 | 不稳定外部调用 |
| 4 | 参数校验函数 | 仅关键字 + isinstance | 入口防脏数据 |
| 5 | 偏函数 + 动态调用 | `functools.partial` | 固化常用配置 |


## 模式 1：配置函数（仅关键字 + `**options`）

- 必传项（host）保持普通参数；可选项全部仅关键字，调用方必须写名字
- 未知配置项经 `**options` 收集，转发给底层驱动 —— 上层不锁死配置面
- 注意点：`**options` 会吞掉拼写错误，若要严格，请加白名单校验（见模式 4）


In [28]:
def connect(host, port=8080, *, timeout=5, retries=3, **options):
    # host 必传；port 可位置可关键字；timeout/retries 强制关键字；其余进 options
    cfg = {"host": host, "port": port, "timeout": timeout, "retries": retries, **options}
    print(cfg)

connect("localhost")                          # 最简调用
connect("localhost", 9090, timeout=10)        # 覆盖端口 + 超时
connect("localhost", retries=5, ssl=True)     # 未知配置 ssl 落入 options 转发


{'host': 'localhost', 'port': 8080, 'timeout': 5, 'retries': 3}
{'host': 'localhost', 'port': 9090, 'timeout': 10, 'retries': 3}
{'host': 'localhost', 'port': 8080, 'timeout': 5, 'retries': 5, 'ssl': True}


## 模式 2：日志函数（`*args` + 仅关键字）

> ⚠️ **修正说明（源文档 13.2）**：源文档把 `**kwargs` 原样转发给 `print()`，
> 调用 `log("错误", level="ERROR", exc_info=True)` 会触发 `TypeError`（`print` 不接受 `exc_info`）。
> 正确做法是把 kwargs 格式化为结构化上下文字段。下面先**真实复现**这个错误。


In [29]:
# 1) 复现源文档的错误写法：把任意 kwargs 原样转发给 print
show_error('print("[ERROR] 消息", exc_info=True)')   # TypeError: 无效关键字参数

# 2) 正确写法：kwargs 格式化为日志上下文
def log(message, *args, level="INFO", **context):
    # message 位置传入；*args 收集补充对象；level 强制可读；context 收集结构化字段
    line = f"[{level}] {message}"
    if args:
        line += " " + " ".join(map(str, args))
    if context:
        line += " (" + " ".join(f"{k}={v}" for k, v in sorted(context.items())) + ")"
    print(line)

log("服务启动")
log("用户登录", "Alice", level="DEBUG")
log("请求失败", level="ERROR", retry=True, attempt=2)


TypeError: print() got an unexpected keyword argument 'exc_info'
[INFO] 服务启动
[DEBUG] 用户登录 Alice
[ERROR] 请求失败 (attempt=2 retry=True)


## 模式 3：带参数的重试装饰器

- 三层结构：`retry(times=...)` 返回 `decorator`，`decorator` 返回 `wrapper`
- `wrapper(*args, **kwargs)` 万能转发，对**任何签名**的函数通用
- `@wraps` 保留被装饰函数的名字与签名
- 注意点：`exceptions` 建议显式收窄（如 `(ValueError,)`），不要捕获所有异常无脑重试


In [30]:
import functools

def retry(times=3, exceptions=(Exception,)):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            for i in range(times):
                try:
                    return func(*args, **kwargs)
                except exceptions:
                    if i == times - 1:   # 最后一次仍失败：向上抛出
                        raise
        return wrapper
    return decorator

attempts = 0
@retry(times=5, exceptions=(ValueError,))
def parse(text):
    # 模拟前两次解析失败、第三次成功（确定性，无随机）
    global attempts
    attempts += 1
    if attempts < 3:
        raise ValueError("暂不可解析")
    return int(text)

print("解析结果:", parse("42"), "| 实际尝试次数:", attempts)


解析结果: 42 | 实际尝试次数: 3


## 模式 4：参数校验函数（仅关键字 + 类型检查）

- 可选项（email）设为仅关键字，调用处自解释
- 校验放函数**最前面**：越早失败，排查成本越低
- 注意点：`isinstance(age, int)` 对 `True` 也成立（bool 是 int 子类），需显式排除


In [31]:
def create_user3(name, age, *, email=None):
    if not isinstance(name, str):
        raise TypeError("name 必须是字符串")
    if not isinstance(age, int) or isinstance(age, bool) or age < 0:
        raise ValueError("age 必须是非负整数")
    print({"name": name, "age": age, "email": email})

create_user3("Alice", 30)
create_user3("Bob", 25, email="bob@example.com")

# 反例：真实触发校验错误
show_error('create_user3("Alice", -1)')
show_error('create_user3("Alice", True)')


{'name': 'Alice', 'age': 30, 'email': None}
{'name': 'Bob', 'age': 25, 'email': 'bob@example.com'}
ValueError: age 必须是非负整数
ValueError: age 必须是非负整数


## 模式 5：偏函数 + 动态调用

- `functools.partial` 固化部分参数（常配合关键字），快速派生特化函数
- `call(func, /, *args, **kwargs)`：`/` 保证 func 只能按位置传，参数区干净无歧义
- 注意点：partial 固化的是**关键字**参数时，原函数对应参数最好也是关键字友好的


In [32]:
from functools import partial

def power3(base, exp):
    return base ** exp

square = partial(power3, exp=2)   # 固化 exp=2
cube = partial(power3, exp=3)     # 固化 exp=3
print(square(5), cube(5))         # 25 125

# 动态调用：func 设为仅位置，其余参数原样转发
def call(func, /, *args, **kwargs):
    return func(*args, **kwargs)

def add5(a, b):
    return a + b
print(call(add5, 1, 2), call(square, 6))


25 125
3 36


# 内置 / 生态对照

> 本章解决什么问题：标准库里有哪些能力与参数系统直接相关，各自何时用。

| 能力 | 模块 | 一句话用途 | 与参数系统的关系 |
|---|---|---|---|
| `partial` | `functools` | 固化部分参数，派生新函数 | 参数偏应用的官方实现 |
| `wraps` | `functools` | 装饰器复制元数据 + `__wrapped__` | 配合 `*args/**kwargs` 保签名 |
| `lru_cache` | `functools` | 官方缓存装饰器 | 替代手写 `_cache={}` 默认参数 |
| `signature` / `Parameter` | `inspect` | 程序化读取签名与参数种类 | 五种参数类型的机器可读枚举 |
| `overload` | `typing` | 声明多套签名给类型检查器 | 同名函数的参数分流 |
| `get_type_hints` | `typing` | 运行时读取注解 | 注解驱动的校验框架基础 |
| `field(kw_only=True)` | `dataclasses`（3.10+） | 数据类字段强制关键字 | `*` 约束在类上的延伸 |


In [33]:
import inspect
from functools import lru_cache, partial

# 1) inspect.Parameter.kind —— 五种参数类型的"官方枚举"，与第 1 章一一对应
def demo_sig(a, b, /, c, d=1, *args, e, f=2, **kwargs): ...
for name, p in inspect.signature(demo_sig).parameters.items():
    print(f"{name:8} kind = {p.kind.name}")

# 2) lru_cache：官方缓存（替代手写默认参数缓存）
@lru_cache(maxsize=None)
def fib3(n):
    return n if n < 2 else fib3(n - 1) + fib3(n - 2)
print("fib3(50) =", fib3(50), "|", fib3.cache_info())


a        kind = POSITIONAL_ONLY
b        kind = POSITIONAL_ONLY
c        kind = POSITIONAL_OR_KEYWORD
d        kind = POSITIONAL_OR_KEYWORD
args     kind = VAR_POSITIONAL
e        kind = KEYWORD_ONLY
f        kind = KEYWORD_ONLY
kwargs   kind = VAR_KEYWORD
fib3(50) = 12586269025 | CacheInfo(hits=48, misses=51, maxsize=None, currsize=51)


# 综合实战：LLM API 客户端封装

> 本章解决什么问题：把前面 12 章的知识点**组合**进一个贴近真实业务的类 —— LLM 服务的客户端封装。

> **场景假设**：原始需求中的业务场景占位符未填写。依据目标目录 `LLMOps/llmops-api`，本例取"**LLM API 服务开发**"作为业务场景：封装一个聊天客户端，涉及配置、URL 拼接、参数透传、重试与校验。

设计蓝图（用到哪些章节）：

| 设计点 | 用到的知识点 |
|---|---|
| `base_url, api_key` 普通参数 | 第 2 章 位置/关键字 |
| `timeout, retries, model` 仅关键字 | 第 8 章 `*` |
| `**extra` 收集未知配置 | 第 5 章 `**kwargs` |
| `retry` 装饰器 + 万能转发 | 第 11 章 |
| `build_url(*paths, **params)` | 第 4/5/10 章 |
| `chat` 内先校验后转发 | 第 12 章 + 模式 4 |


In [34]:
import functools

# ---------- 1) 带参数的重试装饰器（模式 3） ----------
def retry(times=3, exceptions=(Exception,)):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            last_exc = None
            for _ in range(times):
                try:
                    return func(*args, **kwargs)
                except exceptions as e:
                    last_exc = e
            raise last_exc
        return wrapper
    return decorator

# ---------- 2) 参数校验工具（模式 4） ----------
def _require_positive_int(name, value):
    # 错误信息带参数名，方便定位
    if not isinstance(value, int) or isinstance(value, bool) or value <= 0:
        raise ValueError(f"{name} 必须是正整数，得到: {value!r}")

# ---------- 3) 客户端主体 ----------
class LLMClient:
    def __init__(self, base_url, api_key, *, timeout=30, retries=2,
                 model="demo-model", **extra):
        # base_url/api_key：必传，位置或关键字皆可
        # timeout/retries/model：仅关键字，调用处自解释，未来加参数不破坏旧调用
        # **extra：未知配置全部收集，原样透传给底层
        if not isinstance(base_url, str) or not base_url.startswith("http"):
            raise ValueError("base_url 必须是 http(s) 地址")
        _require_positive_int("timeout", timeout)
        _require_positive_int("retries", retries)
        self.base_url = base_url.rstrip("/")
        self.api_key = api_key
        self.timeout = timeout
        self.retries = retries
        self.model = model
        self.extra = extra

    def build_url(self, *paths, **params):
        # *paths 收集路径段；**params 收集查询参数（sorted 保证输出确定）
        url = "/".join([self.base_url, *[p.strip("/") for p in paths]])
        if params:
            query = "&".join(f"{k}={v}" for k, v in sorted(params.items()))
            url += "?" + query
        return url

    @retry(times=3, exceptions=(ConnectionError,))
    def chat(self, prompt, *, temperature=0.7, max_tokens=1024, **extra):
        # prompt 按位置传；采样参数全部仅关键字；未知参数经 **extra 透传
        _require_positive_int("max_tokens", max_tokens)
        if not 0 <= temperature <= 2:
            raise ValueError("temperature 必须在 [0, 2] 之间")
        payload = {"model": self.model, "prompt": prompt,
                   "temperature": temperature, "max_tokens": max_tokens, **extra}
        return self._transport(self.build_url("v1", "chat"), **payload)

    def _transport(self, url, **payload):
        # 桩实现：替代真实 HTTP 调用，保证离线、确定性可复现（修正说明 #2）
        return {"url": url, "echo": payload["prompt"], "params": payload}


In [35]:
# ---------- 4) 使用与验证 ----------
client = LLMClient("https://api.llmops.internal/", "sk-demo",
                   timeout=10, model="llama-3", tags=["prod"])   # tags 落入 extra 透传

print("URL 拼接:", client.build_url("v1", "models", detail=True))
print("正常调用:", client.chat("解释一下 *args", temperature=0.2, top_p=0.9))
print("透传的未知配置 extra:", client.extra)

# 反例：校验真实触发（ValueError 不在 retry 的捕获范围，不会被吞）
show_error('client.chat("hi", temperature=9)')
show_error('LLMClient("ftp://bad", "k")')


URL 拼接: https://api.llmops.internal/v1/models?detail=True
正常调用: {'url': 'https://api.llmops.internal/v1/chat', 'echo': '解释一下 *args', 'params': {'model': 'llama-3', 'prompt': '解释一下 *args', 'temperature': 0.2, 'max_tokens': 1024, 'top_p': 0.9}}
透传的未知配置 extra: {'tags': ['prod']}
ValueError: temperature 必须在 [0, 2] 之间
ValueError: base_url 必须是 http(s) 地址


## 设计取舍说明

| 设计点 | 备选方案 | 为什么这样选 |
|---|---|---|
| `timeout` 等设为仅关键字 | 普通参数 | 防止 `LLMClient(url, key, 5)` 这类"数字含义不明"的调用；后续新增参数不破坏既有调用 |
| `**extra` 收集未知配置 | 显式穷举所有配置 | 配置面随底层 SDK 演进，穷举会频繁改签名；代价是吞拼写错误，由构造期校验兜底 |
| `chat` 里先校验再转发 | 交给底层报错 | 越早失败越好，且错误信息自带参数名，日志可直接定位 |
| `retry` 只捕获 `ConnectionError` | 捕获所有 `Exception` | 参数错误（ValueError）重试没有意义，只应重试"环境性"失败 |
| `_transport` 用桩实现 | 真实 `requests` 调用 | 教学环境需离线、输出确定；生产中将此方法替换为真实 HTTP 即可，签名不变 |


# 常见陷阱汇总：陷阱 / 症状 / 解法

> 本章解决什么问题：把全部易错点压成一张速查表，每个陷阱都有可运行的复现代码。

| # | 陷阱 | 症状 | 解法 |
|---|---|---|---|
| 1 | 可变对象做默认参数 | 多次调用状态累积（`[1]` → `[1,2]`） | 默认 `None`，函数体内新建 |
| 2 | 默认参数在非默认之前 | `SyntaxError` | 调整顺序：非默认在前 |
| 3 | `*args` 后写普通位置参数 | 误以为不合法 / 实际自动变仅关键字 | 明确：`*args` 后的形参自动仅关键字 |
| 4 | 重复传参 | `TypeError: multiple values` | 一个参数只传一次 |
| 5 | `**` 解包非字符串键 | `TypeError: keywords must be strings` | 键必须是字符串 |
| 6 | `if not x` 判默认值 | `0`/`""`/`False` 被误判为"没传" | 用 `x is None` |
| 7 | 位置参数跟在关键字后 | `SyntaxError` | 位置参数放前面 |
| 8 | 可变参数的性能开销 | 极端热点路径额外打包/解包 | 热点函数用显式参数 |
| 9 | `**kwargs` 吞掉拼写错误 | 传错参数名也不报错 | 白名单校验（见第 5 章） |


In [36]:
# 陷阱 1：可变默认参数（复现）
def f14(x, lst=[]):
    lst.append(x)
    return lst
print("连续两次调用:", f14(1), f14(2))   # [1] [1, 2] —— 状态泄漏

# 陷阱 4：重复传参（复现）
show_error('f14(1, x=2)')   # TypeError: multiple values for argument 'x'

# 陷阱 6：not x 误判（复现）
def f15(x=None):
    if not x:
        x = []
    return x
print("f15(0) 被吞成:", f15(0))   # [] —— 0 是合法入参却被替换

# 陷阱 7：顺序错误（复现）
show_error('f14(x=1, 2)')   # SyntaxError

# 陷阱 9：**kwargs 吞错（复现）
def f16(*args, **kwargs):
    return args, kwargs
print("拼错参数名也不报错:", f16(1, 2, 3, wrong_param=1))   # 静默收集


连续两次调用: [1, 2] [1, 2]
TypeError: f14() got multiple values for argument 'x'
f15(0) 被吞成: []
SyntaxError: positional argument follows keyword argument
拼错参数名也不报错: ((1, 2, 3), {'wrong_param': 1})


In [37]:
# 陷阱 3：*args 之后的参数自动成为仅关键字（验证"合法"这一面）
def f_ok(*args, a):          # 合法！a 自动是仅关键字参数
    return args, a
print(f_ok(1, 2, a=3))       # ✅ 正确
show_error('f_ok(1, 2, 3)')  # ❌ a 只能关键字传 —— 这正是"自动仅关键字"的证据

# 陷阱 2/14.2：仅关键字参数漏传（复现）
def g2(a, b=1, *args, c, **kwargs):
    return a, b, args, c, kwargs
show_error('g2(1, 2, 3)')     # TypeError: 缺少仅关键字参数 c
print("正确写法:", g2(1, 2, 3, c=4))

# 陷阱 8：性能开销（确定性对比 —— 用调用次数而非计时）
def explicit(a, b):
    return a + b
def star(*args):
    return args[0] + args[1]
print("两者结果一致:", explicit(1, 2) == star(1, 2))
print("结论：*args 额外打包成元组，热点路径可用显式参数（开销通常可忽略）")


((1, 2), 3)
TypeError: f_ok() missing 1 required keyword-only argument: 'a'
TypeError: g2() missing 1 required keyword-only argument: 'c'
正确写法: (1, 2, (3,), 4, {})
两者结果一致: True
结论：*args 额外打包成元组，热点路径可用显式参数（开销通常可忽略）


# 总结

## 通用代码模板（可直接复制的骨架）


In [38]:
def template(
    pos_only,          # 1) 仅位置参数：内部实现细节，不暴露参数名
    /,                 #    ← / 之前：仅位置
    normal,            # 2) 普通参数：位置或关键字皆可
    default=10,        # 3) 默认参数：只用不可变对象，可变对象用 None 占位
    *args,             # 4) 可变位置参数：收集为 tuple
    kw_only,           # 5) 仅关键字参数：* 之后，强制调用方写参数名
    kw_default=20,     #    仅关键字 + 默认值（可不传）
    **kwargs,          # 6) 可变关键字参数：收集为 dict，用于透传
):
    print(pos_only, normal, default, args, kw_only, kw_default, kwargs)

template(1, 2, 3, 4, kw_only=5, extra="x")


1 2 3 (4,) 5 20 {'extra': 'x'}


## 记忆口诀（5 条）

1. **位置在前，关键字在后；同名参数只传一次**
2. **默认值定义时求值一次；可变默认用 `None`**
3. **`/` 前锁位置，`*` 后锁名字**
4. **`*args` 聚元组，`**kwargs` 聚字典；`**` 的键必须是字符串**
5. **转发就用 `*args, **kwargs`，装饰器加 `@wraps` 保签名**

## 自测练习

尝试独立完成后再看参考实现：

1. **练习 1**：写 `format_price(amount, *, currency="¥", thousand_sep=True)`，返回 `"¥1,234.50"` 样式；非法 `amount` 抛 `TypeError`
2. **练习 2**：写 `safe_call(func, /, *args, ignore_errors=False, **kwargs)` —— `func` 仅位置；出错且 `ignore_errors=True` 时打印并返回 `None`，否则向上抛
3. **练习 3**：不用 `lru_cache`，手写一个 `cached` 装饰器（注意 kwargs 需转成可哈希的 key），并用调用计数验证缓存命中
4. **思考题**（无代码）：`def f(a, b=1, /, c=2, *, d)` 中，`b` 和 `c` 都是"带默认值的参数"，它们的传参约束有什么区别？


In [39]:
# ========== 练习 1 参考实现 ==========
def format_price(amount, *, currency="¥", thousand_sep=True):
    # currency/thousand_sep 仅关键字，调用处自解释
    if not isinstance(amount, (int, float)) or isinstance(amount, bool):
        raise TypeError("amount 必须是数字")
    s = f"{amount:,.2f}" if thousand_sep else f"{amount:.2f}"
    return f"{currency}{s}"

assert format_price(1234.5) == "¥1,234.50"
assert format_price(1234.5, currency="$", thousand_sep=False) == "$1234.50"
show_error('format_price("一百")')
print("练习 1：全部断言通过")

# ========== 练习 2 参考实现 ==========
def safe_call(func, /, *args, ignore_errors=False, **kwargs):
    # func 仅位置 —— 它是"被操作对象"而非业务参数，位置语义最自然
    try:
        return func(*args, **kwargs)
    except Exception as e:
        if not ignore_errors:
            raise
        print(f"[已忽略] {type(e).__name__}: {e}")
        return None

assert safe_call(int, "42") == 42
assert safe_call(int, "x", ignore_errors=True) is None
show_error('safe_call(int, "x")')   # 不忽略时正常向上抛
print("练习 2：全部断言通过")

# ========== 练习 3 参考实现 ==========
import functools

def cached(func):
    _cache = {}                      # 闭包持有缓存，规避默认参数陷阱
    @functools.wraps(func)           # 保留原函数元数据
    def wrapper(*args, **kwargs):
        key = (args, tuple(sorted(kwargs.items())))   # kwargs 排序后可哈希、与顺序无关
        if key not in _cache:
            _cache[key] = func(*args, **kwargs)
        return _cache[key]
    return wrapper

call_count = 0
@cached
def slow_square(n):
    global call_count
    call_count += 1
    return n * n

assert slow_square(9) == 81 and slow_square(9) == 81
assert call_count == 1                 # 第二次命中缓存，未真正执行
print("练习 3：全部断言通过（缓存生效，实际执行 1 次）")


TypeError: amount 必须是数字
练习 1：全部断言通过
[已忽略] ValueError: invalid literal for int() with base 10: 'x'
ValueError: invalid literal for int() with base 10: 'x'
练习 2：全部断言通过
练习 3：全部断言通过（缓存生效，实际执行 1 次）


## 结语

回到开头那句话：

> **参数设计的核心是：让调用者既能按位置传，也能按关键字传；该限制的地方用 `/` 和 `*` 明确约束；转发参数用 `*args` 和 `**kwargs`。**

五个参数段位的完整顺序：`仅位置, /, 普通, *args, 仅关键字, **kwargs` —— 写函数前先想清楚每个参数属于哪一段，签名就自然健壮了。

---

*本 Notebook 所有代码单元均真实执行；错误示例通过 `show_error` 真实触发并捕获，单元本身 0 报错。*
